In [62]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
# ... and any other imports

In [67]:
import pandas as pd
emotion_df = pd.read_csv("../data/emotion_cleaned.csv")

In [64]:
from datasets import load_dataset

In [68]:
dataset = load_dataset("google-research-datasets/go_emotions")

In [69]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [76]:
print(dataset["train"])

Dataset({
    features: ['text', 'labels', 'id'],
    num_rows: 43410
})


In [75]:
print(dataset["train"][0])

{'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej'}


In [74]:
print(dataset["train"].column_names)

['text', 'labels', 'id']


In [73]:
print("Train:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))

Train: 43410
Validation: 5426
Test: 5427


In [77]:
print(dataset["train"].features)

{'text': Value('string'), 'labels': List(ClassLabel(names=['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'])), 'id': Value('string')}


In [80]:
print(dataset["train"].features["labels"])

List(ClassLabel(names=['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']))


In [81]:
label_names = dataset["train"].features["labels"].feature.names

print("Number of emotions:", len(label_names))
print(label_names)

Number of emotions: 28
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [82]:
label_names = dataset["train"].features["labels"].feature.names

print(label_names)
print("Total labels:", len(label_names)) 

['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']
Total labels: 28


In [13]:
from collections import Counter

label_counts = Counter()

for example in dataset["train"]:
    for label in example["labels"]:
        label_counts[label] += 1

for label_id, count in sorted(label_counts.items()):
    print(label_id, "→", label_names[label_id], ":", count)

0 → admiration : 4130
1 → amusement : 2328
2 → anger : 1567
3 → annoyance : 2470
4 → approval : 2939
5 → caring : 1087
6 → confusion : 1368
7 → curiosity : 2191
8 → desire : 641
9 → disappointment : 1269
10 → disapproval : 2022
11 → disgust : 793
12 → embarrassment : 303
13 → excitement : 853
14 → fear : 596
15 → gratitude : 2662
16 → grief : 77
17 → joy : 1452
18 → love : 2086
19 → nervousness : 164
20 → optimism : 1581
21 → pride : 111
22 → realization : 1110
23 → relief : 153
24 → remorse : 545
25 → sadness : 1326
26 → surprise : 1060
27 → neutral : 14219


In [14]:
from collections import Counter

label_per_sample = Counter()

for example in dataset["train"]:
    label_per_sample[len(example["labels"])] += 1

print(label_per_sample)

Counter({1: 36308, 2: 6541, 3: 532, 4: 28, 5: 1})


In [15]:
train_df = dataset["train"].to_pandas()

print(train_df.shape)
print(train_df.head())

(43410, 3)
                                                text labels       id
0  My favourite food is anything I didn't have to...   [27]  eebbqej
1  Now if he does off himself, everyone will thin...   [27]  ed00q6i
2                     WHY THE FUCK IS BAYLESS ISOING    [2]  eezlygj
3                        To make her feel threatened   [14]  ed7ypvh
4                             Dirty Southern Wankers    [3]  ed0bdzj


In [16]:
print(train_df["labels"].head(20).tolist())

[array([27]), array([27]), array([2]), array([14]), array([3]), array([26]), array([15]), array([ 8, 20]), array([0]), array([27]), array([6]), array([1, 4]), array([27]), array([5]), array([3]), array([ 3, 12]), array([15]), array([2]), array([27]), array([ 6, 22])]


In [17]:
single_label_df = train_df[
    train_df["labels"].apply(lambda x: len(x) == 1)
].copy()

print("Single-label samples:", len(single_label_df))

Single-label samples: 36308


In [18]:
single_label_df["label"] = single_label_df["labels"].apply(
    lambda x: label_names[x[0]]
)

In [19]:
print(single_label_df[["text", "label"]].head(10))

                                                 text       label
0   My favourite food is anything I didn't have to...     neutral
1   Now if he does off himself, everyone will thin...     neutral
2                      WHY THE FUCK IS BAYLESS ISOING       anger
3                         To make her feel threatened        fear
4                              Dirty Southern Wankers   annoyance
5   OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe...    surprise
6   Yes I heard abt the f bombs! That has to be wh...   gratitude
8   Damn youtube and outrage drama is super lucrat...  admiration
9   It might be linked to the trust factor of your...     neutral
10  Demographics? I don’t know anybody under 35 wh...   confusion


In [20]:
print(single_label_df["label"].value_counts())

label
neutral           12823
admiration         2710
approval           1873
gratitude          1857
amusement          1652
annoyance          1451
love               1427
disapproval        1402
curiosity          1389
anger              1025
optimism            861
confusion           858
joy                 853
sadness             817
surprise            720
disappointment      709
caring              649
realization         586
excitement          510
disgust             498
fear                430
desire              389
remorse             353
embarrassment       203
relief               88
nervousness          85
pride                51
grief                39
Name: count, dtype: int64


In [21]:
selected_emotions = [
    "admiration",
    "approval",
    "gratitude",
    "amusement",
    "annoyance",
    "love",
    "disapproval",
    "curiosity",
    "anger",
    "optimism",
    "confusion",
    "joy",
    "sadness",
    "surprise",
    "disappointment",
    "caring",
    "realization",
    "excitement"
]

print(len(selected_emotions))

18


In [22]:
emotion_parts = []

for emotion in selected_emotions:
    temp = single_label_df[
        single_label_df["label"] == emotion
    ].sample(n=500, random_state=42)
    
    emotion_parts.append(temp)

In [23]:
neutral_data = single_label_df[
    single_label_df["label"] == "neutral"
].sample(n=1000, random_state=42)

emotion_parts.append(neutral_data)

In [24]:
import pandas as pd

In [25]:
emotion_df = pd.concat(emotion_parts, ignore_index=True)

In [26]:
emotion_df = emotion_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [27]:
emotion_df = emotion_df[["text", "label"]]

In [28]:
print("Dataset shape:", emotion_df.shape)
print(emotion_df["label"].value_counts())

Dataset shape: (10000, 2)
label
neutral           1000
sadness            500
optimism           500
amusement          500
approval           500
confusion          500
admiration         500
love               500
disappointment     500
joy                500
curiosity          500
realization        500
annoyance          500
surprise           500
excitement         500
disapproval        500
caring             500
gratitude          500
anger              500
Name: count, dtype: int64


In [41]:
print(emotion_df.isnull().sum())

text          0
label         0
clean_text    0
dtype: int64


In [37]:

print("Duplicates:", emotion_df.duplicated().sum())

Duplicates: 0


In [83]:
print(emotion_df["text"].str.len().describe())

count    10000.000000
mean        68.305000
std         36.829277
min          4.000000
25%         38.000000
50%         65.000000
75%         95.000000
max        703.000000
Name: text, dtype: float64


In [84]:
print(emotion_df.head(10))

                                                text           label  \
0  He wrote a pretty good piece right after his b...      admiration   
1  It’s only missing a line that says “who was bo...  disappointment   
2  There's a bus to Windsor that I walk by after ...         neutral   
3  If my dog were as ugly as you, id shave its bu...       annoyance   
4  Aye it's doing the rounds so I'm sure I'll see...         neutral   
5  Any man who won’t fight for his kids isn’t wor...     disapproval   
6  Glad to see I'm not the only one with that bug...             joy   
7            He s ugly and short. It's pretty clear.     disapproval   
8                                           Burn it!           anger   
9  Forthwith this group will remain unnamed, veri...        optimism   

                                          clean_text  
0  he wrote a pretty good piece right after his b...  
1  it’s only missing a line that says “who was bo...  
2  there's a bus to windsor that i walk by

In [85]:
emotion_df = emotion_df.drop_duplicates().reset_index(drop=True)

In [86]:
print("Shape after removing duplicates:", emotion_df.shape)

Shape after removing duplicates: (10000, 3)


In [87]:
print(emotion_df["label"].value_counts())

label
neutral           1000
admiration         500
disappointment     500
annoyance          500
disapproval        500
joy                500
anger              500
optimism           500
caring             500
sadness            500
curiosity          500
gratitude          500
love               500
amusement          500
approval           500
excitement         500
realization        500
confusion          500
surprise           500
Name: count, dtype: int64


In [46]:
emotion_df["clean_text"] = emotion_df["text"].str.lower()

In [47]:
print(emotion_df[["text", "clean_text"]].head(10))

                                                text  \
0  All mutations are potentially harmful. Especia...   
1  Enfp male here, I would love to find an INTJ g...   
2  Thank you for doing your service. The world is...   
3  > That shit happens all the time with cabs. No...   
4  I’m hoping so. I either make peace with it or ...   
5  Lol i thought the ratio was 35% or 1/3 of [NAM...   
6  They might if there was something to eat in th...   
7  Just in case anyone wants to know what [NAME] ...   
8  The nature of the flex spot makes it so whoeve...   
9  This is why im happy [NAME] is our owner. I do...   

                                          clean_text  
0  all mutations are potentially harmful. especia...  
1  enfp male here, i would love to find an intj g...  
2  thank you for doing your service. the world is...  
3  > that shit happens all the time with cabs. no...  
4  i’m hoping so. i either make peace with it or ...  
5  lol i thought the ratio was 35% or 1/3 of [nam... 

In [48]:
print("Empty text:", emotion_df["clean_text"].eq("").sum())

Empty text: 0


In [49]:
emotion_df.to_csv("../data/emotion_cleaned.csv", index=False)

In [50]:
print("Emotion dataset saved successfully!")

Emotion dataset saved successfully!


In [51]:
print(emotion_df["label"].value_counts())

label
neutral           999
disapproval       500
amusement         500
curiosity         500
realization       500
confusion         500
sadness           500
approval          500
optimism          499
anger             499
disappointment    499
surprise          499
admiration        499
annoyance         499
joy               498
excitement        498
caring            497
gratitude         496
love              495
Name: count, dtype: int64


In [52]:
target_counts = {
    "optimism": 500,
    "admiration": 500,
    "disappointment": 500,
    "annoyance": 500,
    "surprise": 500,
    "anger": 500,
    "joy": 500,
    "excitement": 500,
    "caring": 500,
    "gratitude": 500,
    "love": 500
}

additional_parts = []

for emotion, target in target_counts.items():
    current = emotion_df[emotion_df["label"] == emotion].shape[0]
    needed = target - current
    
    if needed > 0:
        already_used = set(
            emotion_df[emotion_df["label"] == emotion]["text"]
        )
        
        available = single_label_df[
            (single_label_df["label"] == emotion) &
            (~single_label_df["text"].isin(already_used))
        ]
        
        additional = available.sample(
            n=needed,
            random_state=42
        )
        
        additional_parts.append(
            additional[["text", "label"]]
        )

In [53]:
additional_df = pd.concat(
    additional_parts,
    ignore_index=True
)

print("Additional samples:", len(additional_df))

Additional samples: 22


In [54]:
emotion_df = pd.concat(
    [emotion_df, additional_df],
    ignore_index=True
)

In [55]:
emotion_df = emotion_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [56]:
print("Current shape:", emotion_df.shape)

Current shape: (9999, 3)


In [57]:
print(emotion_df["label"].value_counts().sort_values())

label
admiration        500
disappointment    500
annoyance         500
disapproval       500
anger             500
joy               500
optimism          500
caring            500
love              500
sadness           500
curiosity         500
gratitude         500
approval          500
amusement         500
excitement        500
realization       500
surprise          500
confusion         500
neutral           999
Name: count, dtype: int64


In [58]:
already_used = set(emotion_df["text"])

available_neutral = single_label_df[
    (single_label_df["label"] == "neutral") &
    (~single_label_df["text"].isin(already_used))
]

one_neutral = available_neutral.sample(
    n=1,
    random_state=100
)[["text", "label"]]

emotion_df = pd.concat(
    [emotion_df, one_neutral],
    ignore_index=True
)

In [59]:
print("Final shape:", emotion_df.shape)
print(emotion_df["label"].value_counts())

Final shape: (10000, 3)
label
neutral           1000
admiration         500
disappointment     500
annoyance          500
disapproval        500
joy                500
anger              500
optimism           500
caring             500
sadness            500
curiosity          500
gratitude          500
love               500
amusement          500
approval           500
excitement         500
realization        500
confusion          500
surprise           500
Name: count, dtype: int64


In [60]:
emotion_df.to_csv(
    "../data/emotion_cleaned.csv",
    index=False
)

print("Final emotion dataset saved successfully!")

Final emotion dataset saved successfully!


In [61]:
import os

print(os.path.exists("../data/emotion_cleaned.csv"))

True
